## Extracao - VSOL (Excel, SITE + E-MAIL)

RASCUNHO - ainda nao testado em Databricks real. Ajustar apos a primeira execucao com dado de verdade.

Le o Excel bruto que o Power Automate pousou na pasta do SharePoint (`sharepoint_vsol_bruto_folder`),
filtra so as linhas do `projeto` atual (via `Proposta Comercial`) e grava essa fatia na Bronze
(`BRONZE/vsol/`). Nao faz join com Station nem de-para de parametro -- isso e feito no
`02_limpeza.ipynb` (widget `fonte=vsol`), igual a separacao que ja existe hoje entre `01_extracao_api` e
`02_limpeza`.

**Dois formatos de arquivo, mesma pasta de pouso**: a VSOL entrega dado de duas formas (site/portal e
e-mail semanal), com colunas diferentes entre si -- ver `docs/vsol-integracao.md`, secoes 3.1/3.2. O
widget `formato` (`"auto"`/`"site"`/`"email"`) decide qual branch de leitura roda; em `"auto"` (default),
o notebook detecta sozinho pelo nome das abas do arquivo baixado. Os dois branches convergem pro mesmo
schema canonico antes da gravacao Bronze -- `02_limpeza`/`03_validacoes`/`04_envio_sharepoint` nao
precisam saber qual dos dois formatos gerou o dado.

**Configuracao do Job**: a task deste notebook precisa se chamar `fetch_from_aga_api` na definicao do
Job da VSOL (mesma task key que a Campo usa) -- e assim que `02_limpeza`/`03_validacoes`/
`04_envio_sharepoint` encontram o `output_filename` sem precisar de nenhuma mudanca de codigo neles.
Cada Run desta Job processa **um projeto so** (ver `docs/vsol-integracao.md`, secao 10.2) -- o Power
Automate chama `Run Now` duas vezes por arquivo novo, uma por projeto.

## Setup

In [ ]:
dbutils.library.restartPython()
!pip install --upgrade pip
!pip install openpyxl
!pip install pandas
!pip install python-dotenv
dbutils.library.restartPython()

In [ ]:
import re
import datetime as _dt
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

from config import get_config
from sharepoint_connector import download_file_by_name

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def log_step(etapa, df=None, status="Sucesso", observacoes="", registros_lidos=None, registros_escritos=None):
    n = df.count() if df is not None else 0
    execucao_steps.append({
        "etapa": etapa,
        "status": status,
        "registros_lidos": registros_lidos if registros_lidos is not None else n,
        "registros_escritos": registros_escritos if registros_escritos is not None else n,
        "flags": _collect_flags(df) if df is not None else "\u2014",
        "observacoes": observacoes,
    })

def _collect_flags(df, colunas=None):
    from pyspark.sql import functions as F
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()
        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao - {_log_err}")

## Projeto e arquivo

In [ ]:
dbutils.widgets.text("projeto", "")
dbutils.widgets.text("arquivo", "")
dbutils.widgets.dropdown("formato", "auto", ["auto", "site", "email"])

projeto = dbutils.widgets.get("projeto").strip()
arquivo = dbutils.widgets.get("arquivo").strip()
formato = dbutils.widgets.get("formato").strip().lower()

cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

if not arquivo:
    raise ValueError(
        "Widget 'arquivo' vazio - informe o nome do arquivo pousado na pasta VSOL do SharePoint "
        "(cfg['sharepoint_vsol_bruto_folder'])."
    )

# O campo "Nome"/"Name" do gatilho de arquivo do SharePoint no Power Automate vem sem extensao
# (ex.: "samples-14_09_2026" em vez de "samples-14_09_2026.xlsx") -- completa aqui em vez de
# depender de quem monta o Flow lembrar disso.
if not arquivo.lower().endswith((".xlsx", ".xls")):
    arquivo = arquivo + ".xlsx"

In [ ]:
project_name = projeto
project_path = "/mnt/wst/" + project_name

folder_bronze = project_path + "/BRONZE/"
vsol_path = folder_bronze + "vsol/"

# dbutils.fs.mkdirs(vsol_path)

### Download do Excel (pasta de dados brutos VSOL no SharePoint)

In [ ]:
local_path = f"/tmp/{arquivo}"

result = download_file_by_name(
    folder_path=cfg["sharepoint_vsol_bruto_folder"],
    filename=arquivo,
    local_path=local_path,
)

if result["status"] != "success":
    execucao_steps.append({
        "etapa": "Download do arquivo VSOL",
        "status": "Erro",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": f"Falha ao baixar '{arquivo}': {result['error']}",
    })
    persistir_log()
    raise RuntimeError(f"Falha ao baixar '{arquivo}' de '{cfg['sharepoint_vsol_bruto_folder']}': {result['error']}")

execucao_steps.append({
    "etapa": "Download do arquivo VSOL",
    "status": "Sucesso",
    "registros_lidos": 0,
    "registros_escritos": 0,
    "flags": "\u2014",
    "observacoes": f"'{arquivo}' baixado ({result['size_bytes']} bytes) para {local_path}",
})

### Deteccao do formato do arquivo (SITE ou E-MAIL)

O widget `formato` aceita `"auto"` (default), `"site"` ou `"email"`. Em `"auto"`, decide pelo nome das
abas do arquivo baixado -- se tiver `SYS_Sample` e `SYS_SampleAnalysis`, e E-MAIL; senao, SITE (aba unica
achatada). Ver `docs/vsol-integracao.md`, secoes 3.1/3.2.

In [ ]:
_abas = pd.ExcelFile(local_path).sheet_names

if formato == "auto":
    formato_resolvido = "email" if {"SYS_Sample", "SYS_SampleAnalysis"}.issubset(_abas) else "site"
elif formato in ("site", "email"):
    formato_resolvido = formato
else:
    execucao_steps.append({
        "etapa": "Deteccao de formato",
        "status": "Erro",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": f"Valor invalido para o widget 'formato': '{formato}' (esperado: auto, site ou email)",
    })
    persistir_log()
    raise ValueError(f"Widget 'formato' invalido: '{formato}'")

formato = formato_resolvido

execucao_steps.append({
    "etapa": "Deteccao de formato",
    "status": "Sucesso",
    "registros_lidos": 0,
    "registros_escritos": 0,
    "flags": "\u2014",
    "observacoes": f"formato='{formato}' (abas do arquivo: {_abas})",
})

### Leitura e filtro por projeto

Layout SITE: aba unica, achatada (uma linha por amostra x parametro). Layout E-MAIL: duas abas
relacionadas (`SYS_Sample` + `SYS_SampleAnalysis`), join por `Cod Amostra Lab`. Ver
`docs/vsol-integracao.md` secoes 3.1/3.2/4/14 para o estudo completo dos dois formatos.

As duas branches terminam definindo o mesmo conjunto de nomes de coluna "comuns"
(`COL_PROPOSTA`, `COL_NOME_AMOSTRA`, `COL_PARAM`, `COL_MATRIZ`, `COL_IDAMOSTRA`, `COL_DATA_COLETA`,
`COL_DATA_RECEB`, `COL_RESULTADO`, `COL_UNIDADE`) -- as celulas seguintes usam so esses nomes, sem
precisar saber se o dado veio do SITE ou do E-MAIL.

In [ ]:
if formato == "site":
    df_raw = pd.read_excel(local_path, sheet_name=0, dtype=str)
    df_raw.columns = [c.strip() for c in df_raw.columns]

    COL_PROPOSTA = "Proposta Comercial"
    COL_IDENT = "Identificacao" if "Identificacao" in df_raw.columns else "Identificação"
    COL_TIPO = "Tipo de Amostra"
    COL_CODIGO = "Codigo da Amostra" if "Codigo da Amostra" in df_raw.columns else "Código da Amostra"
    COL_DATA_COLETA = "Data de Coleta"
    COL_DATA_RECEB = "Data de Recebimento"
    COL_DATA_PUB = "Data de Publicacao" if "Data de Publicacao" in df_raw.columns else "Data de Publicação"
    COL_SITUACAO = "Situacao da Amostra" if "Situacao da Amostra" in df_raw.columns else "Situação da Amostra"
    COL_IDENT_ANALISE = "Identificacao da Analise" if "Identificacao da Analise" in df_raw.columns else "Identificação da Análise"
    COL_RESULTADO = "Resultado da Analise" if "Resultado da Analise" in df_raw.columns else "Resultado da Análise"
    COL_UNIDADE = "Unidade de Medida da Analise" if "Unidade de Medida da Analise" in df_raw.columns else "Unidade de Medida da Análise"

    colunas_esperadas = [COL_PROPOSTA, COL_IDENT, COL_TIPO, COL_CODIGO, COL_DATA_COLETA, COL_DATA_RECEB,
                         COL_DATA_PUB, COL_SITUACAO, COL_IDENT_ANALISE, COL_RESULTADO, COL_UNIDADE]
    faltando = [c for c in colunas_esperadas if c not in df_raw.columns]
    if faltando:
        execucao_steps.append({
            "etapa": "Leitura do Excel",
            "status": "Erro",
            "registros_lidos": 0,
            "registros_escritos": 0,
            "flags": "\u2014",
            "observacoes": f"Colunas esperadas nao encontradas no arquivo: {faltando}. Layout pode ter mudado.",
        })
        persistir_log()
        raise RuntimeError(f"Colunas esperadas nao encontradas: {faltando}")

    execucao_steps.append({
        "etapa": "Leitura do Excel",
        "status": "Sucesso",
        "registros_lidos": len(df_raw),
        "registros_escritos": len(df_raw),
        "flags": "\u2014",
        "observacoes": f"{len(df_raw)} linhas lidas de '{arquivo}' (SITE)",
    })

    # Nomes de coluna comuns entre os dois formatos -- daqui pra frente as celulas seguintes usam so
    # estes nomes, sem precisar saber se o dado veio do SITE ou do E-MAIL.
    COL_NOME_AMOSTRA = COL_IDENT
    COL_PARAM = COL_IDENT_ANALISE
    COL_MATRIZ = COL_TIPO
    COL_IDAMOSTRA = COL_CODIGO

elif formato == "email":
    df_sample = pd.read_excel(local_path, sheet_name="SYS_Sample", dtype=str)
    df_sample.columns = [c.strip() for c in df_sample.columns]
    df_analysis = pd.read_excel(local_path, sheet_name="SYS_SampleAnalysis", dtype=str)
    df_analysis.columns = [c.strip() for c in df_analysis.columns]

    COL_PROPOSTA = "Proposta Comercial"
    COL_NOME_AMOSTRA = "Nome Amostra"
    COL_PARAM = "Cod Parametro"
    COL_MATRIZ = "Matriz Monit"
    COL_IDAMOSTRA = "Cod Amostra Lab"
    COL_DATA_COLETA = "Data Coleta"
    COL_DATA_RECEB = "Data Recebimento Lab"
    COL_RESULTADO = "Resultado Original"
    COL_UNIDADE = "Unidade Original"

    colunas_sample_esperadas = ["Codigo Ponto", COL_NOME_AMOSTRA, COL_IDAMOSTRA, COL_DATA_COLETA,
                                 COL_MATRIZ, COL_DATA_RECEB]
    colunas_analysis_esperadas = [COL_IDAMOSTRA, COL_PARAM, COL_RESULTADO, COL_UNIDADE,
                                   "Data e Hora Analise", "Metodo Analise", "Tipo Analise", "LQ"]
    faltando = (
        [c for c in colunas_sample_esperadas if c not in df_sample.columns]
        + [c for c in colunas_analysis_esperadas if c not in df_analysis.columns]
    )
    if faltando:
        execucao_steps.append({
            "etapa": "Leitura do Excel",
            "status": "Erro",
            "registros_lidos": 0,
            "registros_escritos": 0,
            "flags": "\u2014",
            "observacoes": f"Colunas esperadas nao encontradas (E-MAIL): {faltando}. Layout pode ter mudado.",
        })
        persistir_log()
        raise RuntimeError(f"Colunas esperadas nao encontradas (E-MAIL): {faltando}")

    if COL_PROPOSTA not in df_sample.columns:
        execucao_steps.append({
            "etapa": "Leitura do Excel",
            "status": "Erro",
            "registros_lidos": 0,
            "registros_escritos": 0,
            "flags": "\u2014",
            "observacoes": (
                "Arquivo E-MAIL sem a coluna 'Proposta Comercial' em SYS_Sample -- o laboratorio ainda "
                "nao adicionou essa coluna (ver docs/vsol-integracao.md, secao 3.2/13). Nao e possivel "
                "resolver o projeto sem ela."
            ),
        })
        persistir_log()
        raise RuntimeError("Coluna 'Proposta Comercial' ausente em SYS_Sample (formato E-MAIL)")

    execucao_steps.append({
        "etapa": "Leitura do Excel",
        "status": "Sucesso",
        "registros_lidos": len(df_sample) + len(df_analysis),
        "registros_escritos": len(df_sample) + len(df_analysis),
        "flags": "\u2014",
        "observacoes": f"{len(df_sample)} amostra(s) / {len(df_analysis)} resultado(s) lidos de '{arquivo}' (E-MAIL)",
    })

### Filtro de campanha (projeto atual)

In [ ]:
# Reaproveita cfg["campanha_api"] (ja cadastrado pra Campo) como substring de busca -- os valores
# observados em `Proposta Comercial` sempre comecam com esse mesmo texto
# (ex.: "Projeto_1233_IC_CDM_CODEMIN/GO" contem "Projeto_1233_IC_CDM"). Mesma logica pras duas fontes.
campanha_busca = cfg["campanha_api"]

if formato == "site":
    registros_antes_filtro = len(df_raw)
    df_projeto = df_raw[df_raw[COL_PROPOSTA].str.contains(re.escape(campanha_busca), na=False)].copy()
    registros_depois_filtro = len(df_projeto)
elif formato == "email":
    # Filtra as amostras (SYS_Sample) pela campanha antes do join -- resultados (SYS_SampleAnalysis)
    # de amostras de outro projeto caem fora naturalmente no merge (how="inner").
    df_sample_projeto = df_sample[df_sample[COL_PROPOSTA].str.contains(re.escape(campanha_busca), na=False)].copy()
    registros_antes_filtro = len(df_analysis)
    df_projeto = df_analysis.merge(df_sample_projeto, on=COL_IDAMOSTRA, how="inner", suffixes=("", "_amostra"))
    registros_depois_filtro = len(df_projeto)

execucao_steps.append({
    "etapa": "Filtro de campanha",
    "status": "Sucesso" if registros_depois_filtro > 0 else "Aviso",
    "registros_lidos": registros_antes_filtro,
    "registros_escritos": registros_depois_filtro,
    "flags": "\u2014",
    "observacoes": (
        f"{registros_depois_filtro} de {registros_antes_filtro} linhas pertencem a '{campanha_busca}'"
        if registros_depois_filtro
        else f"Nenhuma linha do arquivo pertence a '{campanha_busca}' -- arquivo pode ser so do outro projeto"
    ),
})

if df_projeto.empty:
    persistir_log()
    dbutils.notebook.exit(f"Nenhuma linha de '{campanha_busca}' em '{arquivo}'")

### Filtro de amostras pendentes (sem resultado ainda)

`Situacao da Amostra == "Recebida"` = amostra recebida no lab mas ainda sem resultado publicado
(ver `docs/vsol-integracao.md`, secao 14). Nao sao erro, sao pendentes -- ficam de fora desta carga e
entram numa proxima execucao quando o resultado sair. **So existe no formato SITE** -- o E-MAIL nao tem
coluna de situacao (o laboratorio so envia o e-mail quando ja tem resultado).

In [ ]:
if formato == "site":
    registros_antes_pendente = len(df_projeto)
    df_projeto = df_projeto[df_projeto[COL_SITUACAO] != "Recebida"].copy()
    registros_pendentes = registros_antes_pendente - len(df_projeto)
    execucao_steps.append({
        "etapa": "Filtro de amostras pendentes",
        "status": "Sucesso",
        "registros_lidos": registros_antes_pendente,
        "registros_escritos": len(df_projeto),
        "flags": "\u2014",
        "observacoes": f"{registros_pendentes} linha(s) 'Recebida' (sem resultado ainda) deixadas de fora desta carga",
    })
else:
    execucao_steps.append({
        "etapa": "Filtro de amostras pendentes",
        "status": "Sucesso",
        "registros_lidos": len(df_projeto),
        "registros_escritos": len(df_projeto),
        "flags": "\u2014",
        "observacoes": "Nao aplicavel ao formato E-MAIL -- essa fonte nao tem coluna de situacao/pendencia",
    })

if df_projeto.empty:
    persistir_log()
    dbutils.notebook.exit("Todas as linhas do projeto estavam pendentes (Situacao == Recebida)")

### Checagem de amostra/parametro unico

Uma amostra nao pode ter o mesmo parametro reportado duas vezes com valores diferentes -- isso e erro de
dado do laboratorio, nao algo que o pipeline deva decidir sozinho qual valor manter (ver
`docs/vsol-integracao.md`, secao 3.2 -- caso `CAC4567_2026 REV.xlsx`, onde o mesmo parametro apareceu
com dois valores diferentes pra mesma amostra sob codigos de lab diferentes). Roda pras duas fontes
(nao faz mal nenhum no SITE, que ate hoje nunca mostrou esse padrao nos arquivos analisados). Quando
encontra um par amostra/parametro com resultados divergentes, exclui so essas linhas da carga e loga um
aviso acionavel com o detalhe -- o resto do arquivo segue normalmente.

In [ ]:
_chave = [COL_NOME_AMOSTRA, COL_PARAM]
_n_valores = df_projeto.groupby(_chave)[COL_RESULTADO].transform("nunique")
_conflito = df_projeto[_n_valores > 1]

if not _conflito.empty:
    _pares = _conflito[_chave].drop_duplicates()
    _detalhe = "; ".join(f"{a} / {p}" for a, p in _pares.itertuples(index=False))
    df_projeto = df_projeto[_n_valores <= 1].copy()
    execucao_steps.append({
        "etapa": "Checagem de amostra/parametro unico",
        "status": "Aviso",
        "registros_lidos": len(df_projeto) + len(_conflito),
        "registros_escritos": len(df_projeto),
        "flags": "\u2014",
        "observacoes": (
            f"ERRO DE DADO NO LABORATORIO: {len(_pares)} par(es) amostra/parametro com resultados "
            f"divergentes entre codigos de lab diferentes -- excluidos desta carga, precisam de revisao "
            f"da VSOL: {_detalhe}"
        ),
    })
else:
    execucao_steps.append({
        "etapa": "Checagem de amostra/parametro unico",
        "status": "Sucesso",
        "registros_lidos": len(df_projeto),
        "registros_escritos": len(df_projeto),
        "flags": "\u2014",
        "observacoes": "Nenhum par amostra/parametro com resultados divergentes",
    })

if df_projeto.empty:
    persistir_log()
    dbutils.notebook.exit("Todas as linhas ficaram de fora apos a checagem de amostra/parametro unico")

### De-para de matriz (codigos curtos do E-MAIL)

O E-MAIL usa codigos curtos em `Matriz Monit` (`SO`, `ASUB`, `LNAPL`...) em vez do texto por extenso que
o SITE/API ja usam (`Solo`, `Água Subterrânea`). Mapeia os codigos conhecidos; qualquer codigo
desconhecido (ex. `LNAPL`, que ainda nao tem categoria definida no schema/config) passa sem alteracao e
fica registrado como aviso -- nao inventa de-para sem confirmacao de negocio (ver
`docs/vsol-integracao.md`, secao 3.2).

In [ ]:
_DEPARA_MATRIZ_EMAIL = {
    "SO": "Solo",
    "ASUB": "Água Subterrânea",
}

if formato == "email":
    _matriz_antes = df_projeto[COL_MATRIZ].copy()
    df_projeto[COL_MATRIZ] = df_projeto[COL_MATRIZ].map(lambda m: _DEPARA_MATRIZ_EMAIL.get(m, m))
    _nao_mapeados = sorted(set(_matriz_antes[~_matriz_antes.isin(_DEPARA_MATRIZ_EMAIL.keys())].dropna().unique()))
    execucao_steps.append({
        "etapa": "De-para de matriz",
        "status": "Sucesso" if not _nao_mapeados else "Aviso",
        "registros_lidos": len(df_projeto),
        "registros_escritos": len(df_projeto),
        "flags": "\u2014",
        "observacoes": (
            "Todos os codigos de matriz mapeados" if not _nao_mapeados
            else f"Codigo(s) de matriz sem de-para (mantidos como vieram, revisar antes de usar em escopo/QAQC): {_nao_mapeados}"
        ),
    })
else:
    execucao_steps.append({
        "etapa": "De-para de matriz",
        "status": "Sucesso",
        "registros_lidos": len(df_projeto),
        "registros_escritos": len(df_projeto),
        "flags": "\u2014",
        "observacoes": "Nao aplicavel ao formato SITE -- matriz ja vem por extenso",
    })

### Parsing do resultado (qualificador + valor)

~85% dos resultados chegam como texto com qualificador embutido (ex.: `"< 0,0030"` no SITE,
`"< 250.00"` no E-MAIL). Extrai o qualificador (`<`/`>`) separado do valor -- a mesma regex aceita ponto
e virgula, entao roda igual pras duas fontes via `COL_RESULTADO`. **Nao deriva `limiteQuantificacao`/LQ
daqui** -- no SITE essa informacao nao existe; no E-MAIL ela ja vem pronta na coluna `LQ` e e mapeada
direto na padronizacao (proxima celula).

In [ ]:
_PATTERN_QUALIFICADOR = re.compile(r"^([<>])\s*(.+)$")
_PATTERN_NUMERO = re.compile(r"^-?\d+([.,]\d+)?$")

def _parse_resultado(raw):
    # Retorna (qualifier, resultadoNumerico, resultadoTexto) -- numerico e texto sao mutuamente
    # exclusivos: preenche um ou outro, nunca os dois (bug corrigido em 2026-09-14: resultadoTexto
    # estava vindo preenchido mesmo quando o valor era um numero puro).
    if pd.isna(raw):
        return (None, None, None)
    texto = str(raw).strip()
    m = _PATTERN_QUALIFICADOR.match(texto)
    qualifier = m.group(1) if m else None
    valor = m.group(2).strip() if m else texto
    if _PATTERN_NUMERO.match(valor):
        return (qualifier, valor, None)  # numero (com ou sem qualificador) -> resultadoNumerico
    return (qualifier, None, texto)  # nao e numero -> resultadoTexto

_parsed = df_projeto[COL_RESULTADO].map(_parse_resultado)
df_projeto["qualifier"] = _parsed.map(lambda t: t[0])
df_projeto["resultadoNumerico"] = _parsed.map(lambda t: t[1])  # comma->dot e cast ficam pro 02_limpeza.ipynb (fonte=vsol)
df_projeto["resultadoTexto"] = _parsed.map(lambda t: t[2])

qtd_qualificado = df_projeto["qualifier"].notna().sum()
execucao_steps.append({
    "etapa": "Parsing de resultado",
    "status": "Sucesso",
    "registros_lidos": len(df_projeto),
    "registros_escritos": len(df_projeto),
    "flags": "\u2014",
    "observacoes": f"{qtd_qualificado} de {len(df_projeto)} resultados com qualificador (<, >)",
})

### Padronizacao para o schema canonico (Bronze)

Mesmos nomes de campo que `01_extracao_api` ja produz (`parse_hga_api_json_to_df`), pra
`02_limpeza`/`03_validacoes`/`04_envio_sharepoint` nao precisarem saber a origem do dado. Ver
`docs/vsol-integracao.md` secao 4 pro mapeamento linha a linha dos dois formatos.

Colunas que um formato tem e o outro nao ficam vazias de proposito (`None`) em vez de deixar de existir,
pra bater com a relacao completa de colunas que vai pro banco. O E-MAIL preenche 3 campos que o SITE
nunca preencheu (`metodoAnalise`, `limiteQuantificacao`, `dataHoraAnalise`) -- ver secao 4 do doc.

In [ ]:
if formato == "site":
    df_bronze = pd.DataFrame({
        "campanha": df_projeto[COL_PROPOSTA],
        "idAmostra": df_projeto[COL_IDAMOSTRA].astype(str),
        "nomeAmostra": df_projeto[COL_NOME_AMOSTRA],
        "descricaoAmostra": df_projeto[COL_NOME_AMOSTRA],
        "matriz": df_projeto[COL_MATRIZ],
        "dataHoraAmostragem": df_projeto[COL_DATA_COLETA],
        "dataRecebLab": df_projeto[COL_DATA_RECEB],
        "dataEnvioLab": None,       # nao existe no SITE
        "dataHoraAnalise": None,    # nao existe no SITE
        "dataLiberacao": df_projeto[COL_DATA_PUB],
        "laboratorio": "VSOL",      # nao existe coluna de lab no SITE -- assumido conforme decisao do Sinderley
        "parametroOriginal": df_projeto[COL_PARAM],
        "resultadoOriginal": df_projeto[COL_RESULTADO],
        "resultadoNumerico": df_projeto["resultadoNumerico"],
        "resultadoTexto": df_projeto["resultadoTexto"],
        "qualifier": df_projeto["qualifier"],
        "unidadeOriginal": df_projeto[COL_UNIDADE],
        "OrigemArquivo": "VSOL_SITE",
        # Colunas do schema final (api_validado) que o SITE nao tem hoje -- ficam vazias de proposito, em
        # vez de deixar de existir, pra bater com a relacao completa de colunas que vao pro banco.
        "codigoQualidade": None,     # nao existe no SITE
        "frequencia": None,          # nao existe no SITE
        "metodoAnalise": None,       # nao existe no SITE -- ver docs/vsol-integracao.md
        "limiteQuantificacao": None, # nao existe no SITE -- ver docs/vsol-integracao.md
        "comentario_parametro": None,  # nao existe no SITE
        "comentario_amostra": None,    # nao existe no SITE
        "tipoAnalise": None,           # nao existe no SITE
    })
elif formato == "email":
    df_bronze = pd.DataFrame({
        "campanha": df_projeto[COL_PROPOSTA],
        "idAmostra": df_projeto[COL_IDAMOSTRA].astype(str),
        "nomeAmostra": df_projeto[COL_NOME_AMOSTRA],
        "descricaoAmostra": df_projeto[COL_NOME_AMOSTRA],
        "matriz": df_projeto[COL_MATRIZ],
        "dataHoraAmostragem": df_projeto[COL_DATA_COLETA],
        "dataRecebLab": df_projeto[COL_DATA_RECEB],
        "dataEnvioLab": df_projeto.get("Data Envio Lab"),  # 100% vazio nos arquivos observados, mas existe no schema
        "dataHoraAnalise": df_projeto["Data e Hora Analise"],  # novo -- SITE nunca preencheu
        "dataLiberacao": None,       # nao existe equivalente no E-MAIL
        "laboratorio": "VSOL",
        "parametroOriginal": df_projeto[COL_PARAM],
        "resultadoOriginal": df_projeto[COL_RESULTADO],
        "resultadoNumerico": df_projeto["resultadoNumerico"],
        "resultadoTexto": df_projeto["resultadoTexto"],
        "qualifier": df_projeto["qualifier"],
        "unidadeOriginal": df_projeto[COL_UNIDADE],
        "OrigemArquivo": "VSOL_EMAIL",
        "codigoQualidade": df_projeto.get("Tipo Qualidade"),
        "frequencia": None,          # nao existe no E-MAIL
        "metodoAnalise": df_projeto["Metodo Analise"],  # novo -- SITE nunca preencheu
        "limiteQuantificacao": df_projeto["LQ"],  # novo -- vem com virgula decimal, cast fica pro 02_limpeza.ipynb
        "comentario_parametro": df_projeto.get("Comentario"),
        "comentario_amostra": df_projeto.get("Comentario_amostra"),
        "tipoAnalise": df_projeto["Tipo Analise"],  # novo -- "Inicial"/"Reanalise", so rastreabilidade
    })

# Todas as colunas como string, igual ao schema que 01_extracao_api ja usa (StructType de StringType) --
# 02_limpeza.ipynb (widget fonte=vsol) faz os casts (data, numero) que forem necessarios.
df_bronze = df_bronze.astype(object).where(pd.notnull(df_bronze), None)
df_bronze = df_bronze.astype(str).where(df_bronze.notnull(), None)

### Gravacao Bronze (Delta)

In [ ]:
schema = StructType([StructField(c, StringType(), True) for c in df_bronze.columns])
df_spark = spark.createDataFrame(df_bronze, schema=schema)

_prefixo_pasta = "vsolSITE" if formato == "site" else "vsolEMAIL"
nome_pasta = f"{_prefixo_pasta}_{_dt.datetime.now().strftime('%Y%m%d_%H%M')}"
caminho_completo = f"{vsol_path}{nome_pasta}"

try:
    df_spark.write.format("delta").mode("overwrite").save(caminho_completo)
    execucao_steps.append({
        "etapa": "Gravacao Bronze",
        "status": "Sucesso",
        "registros_lidos": df_spark.count(),
        "registros_escritos": df_spark.count(),
        "flags": "\u2014",
        "observacoes": f"Salvo em {caminho_completo}",
    })
except Exception as _e:
    execucao_steps.append({
        "etapa": "Gravacao Bronze",
        "status": "Erro",
        "registros_lidos": df_spark.count(),
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": str(_e),
    })
    persistir_log()
    raise

print("Nome da pasta de salvamento:", nome_pasta)

In [ ]:
display(pd.DataFrame(execucao_steps))

In [ ]:
persistir_log()

In [ ]:
dbutils.jobs.taskValues.set(
    key="output_filename",
    value=nome_pasta,
)